In [1]:
! pip install torch
! pip install transformers
! pip install re
! git clone https://huggingface.co/ismail071/poetry_model

ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re
Cloning into 'poetry_model'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 12 (delta 0), reused 0 (delta 0), pack-reused 3 (from 1)
Unpacking objects: 100% (12/12), 523.47 KiB | 1.98 MiB/s, done.


In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import re

/Users/ismail/MAI-THWS/NLP-portfolios/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# ============================================================
# Text Cleaning Helpers
# ============================================================
def clean_text(text: str) -> str:
    # Remove ( ... ) and content inside
    text = re.sub(r"\([^)]*\)", "", text)

    # Remove [ ... ] and content inside
    text = re.sub(r"\[[^\]]*\]", "", text)

    # Remove duplicate lines
    lines = text.split("\n")
    seen = set()
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped and stripped not in seen:
            cleaned_lines.append(stripped)
            seen.add(stripped)

    # Rejoin cleaned lines
    text = "\n".join(cleaned_lines)

    return text.strip()


# ============================================================
# Load fine-tuned poetry model
# ============================================================
MODEL_PATH = "poetry_model0"

print("=> Loading fine-tuned poetry model...")
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)
model = GPT2LMHeadModel.from_pretrained(MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("=> Model loaded successfully!")


# ============================================================
# Generate poem from 3 prompt words
# ============================================================
def generate_poem(prompt_words, max_length=520, temperature=0.9, top_p=0.92):
    """
    Generate a poem given three prompt words.
    """
    prompt = f"Write a poem about: {prompt_words}\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove the prompt prefix
    poem = generated_text[len(prompt) :].strip()

    # >>> NEW: clean the poem <<<
    poem = clean_text(poem)

    return poem


# ============================================================
# CLI interface
# ============================================================
if __name__ == "__main__":
    print("\n🎤 Welcome to the Poetry Slam Generator!")
    print("Enter three words to inspire your poem (e.g. 'rain love memory').")
    print("Type 'exit' to quit.\n")

    while True:
        user_input = input("🪶 Enter 3 words: ").strip()

        if user_input.lower() == "exit":
            print("👋 Goodbye!")
            break

        if len(user_input.split()) < 3:
            print("⚠️ Please enter at least three words.")
            continue

        print("\n✨ Generating your poem...\n")
        poem = generate_poem(user_input)
        print(poem)
        print("\n" + "=" * 60 + "\n")


=> Loading fine-tuned poetry model...
=> Model loaded successfully!

🎤 Welcome to the Poetry Slam Generator!
Enter three words to inspire your poem (e.g. 'rain love memory').
Type 'exit' to quit.


✨ Generating your poem...

I watched from my bedroom window as my sister and I,
a big-boned blonde waitress who had been shot in the leg,
dressed in black with a gray hoodie and a turtleneck.
We sat down on two couches to chat, and we talked
about gun rights, but most of all about how
I should be more careful when putting arms around women like this because they’re vulnerable
and need protection—so that if anyone wants a weapon, it’s you. It’s why she let us walk across her bed
with her children in their hands. And so is the mother who was trying not kill me,
who saved my life for three months before killing herself.
The nurse explained to him that our son suffered from schizophrenia
and asked whether he could get his medication now, saying no. We were both afraid
he wouldn't come home safel